# Day-ahead consumption forecast — corrected version

Same goal as `mock_01_consumption_forecast.ipynb`. Each fix is marked with **Fix N** and refers to
the numbering in `mock_01_solution.md`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

pd.set_option("display.width", 120)

## Load data

In [2]:
df = pd.read_csv("../../data/hourly_power_raw.csv")
print(df.shape)
df.dtypes

(17457, 7)


time                object
consumption_mwh    float64
temp_c             float64
wind_ms            float64
solar_wm2          float64
price_eur_mwh       object
region              object
dtype: object

**Fix 1 / 2 / 3.** Parse as UTC, sort, and *drop duplicated timestamps* before anything positional
(`shift`, `rolling`). Then reindex to a complete hourly grid so that `shift(24)` really is 24 hours.

In [3]:
df["time"] = pd.to_datetime(df["time"], utc=True)
df = df.sort_values("time")

print("duplicated timestamps:", df["time"].duplicated().sum())
df = df.drop_duplicates(subset="time", keep="first").set_index("time").drop(columns="region")

full_index = pd.date_range(df.index.min(), df.index.max(), freq="h", tz="UTC")
print("missing hours before reindex:", len(full_index) - len(df))
df = df.reindex(full_index)
assert df.index.is_monotonic_increasing and df.index.is_unique

duplicated timestamps: 15
missing hours before reindex: 78


**Fix 4.** `describe()` shows `temp_c` min = -999: a sentinel, not a temperature. Set to NaN *before*
any fill, otherwise the fill propagates it.  
**Fix 5.** Price `"missing"` → NaN, not 0. Zero is a real price in this market (negative prices exist),
so `fillna(0)` invents observations. Short gaps are forward-filled with a limit; the rest stay NaN and are dropped.

In [4]:
print("temp sentinels:", (df["temp_c"] <= -100).sum())
df.loc[df["temp_c"] <= -100, "temp_c"] = np.nan

df["price_eur_mwh"] = pd.to_numeric(df["price_eur_mwh"], errors="coerce")
print("price missing after coercion:", df["price_eur_mwh"].isna().sum())

df["temp_c"] = df["temp_c"].interpolate(limit=3)
df["price_eur_mwh"] = df["price_eur_mwh"].ffill(limit=3)
df.describe().round(1)

temp sentinels: 63
price missing after coercion: 178


,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh
count,17442.0,17499.0,17442.0,17442.0,17499.0
mean,29314.9,9.9,7.3,97.7,98.5
std,4207.4,6.7,2.5,155.3,36.9
min,18092.9,-6.4,0.0,0.0,-19.9
25%,26444.7,4.5,5.6,0.0,73.4
50%,29670.6,10.0,7.2,0.0,97.6
75%,32372.8,15.3,9.0,143.1,122.8
max,40824.9,27.7,16.0,794.6,419.6


**Fix 6.** The original called `index.hour` "local hour" on a tz-naive index that is actually UTC.
GB local time is UTC+1 for seven months of the year, so the evening peak moves by one hour between
winter and summer in UTC. Convert to `Europe/London` before extracting calendar features.

In [5]:
local = df.index.tz_convert("Europe/London")
df["hour"] = local.hour
df["dow"] = local.dayofweek

profile = df.groupby(["hour", "dow"])["consumption_mwh"].mean().unstack()
profile.round(0).head(6)

dow,0,1,2,3,4,5,6
hour,,,,,,,
0,25312.0,26610.0,26686.0,26676.0,26523.0,25809.0,24622.0
1,25618.0,25602.0,25663.0,25703.0,25505.0,23509.0,23573.0
2,24797.0,24733.0,24769.0,24871.0,24680.0,22773.0,22652.0
3,24387.0,24336.0,24466.0,24391.0,24192.0,22378.0,22294.0
4,24354.0,24324.0,24334.0,24317.0,24162.0,22303.0,22287.0
5,25026.0,24893.0,24875.0,24933.0,24780.0,22890.0,22872.0


**Fix 7.** `lag168 = shift(-168)` was a sign typo: it is consumption 168 hours in the *future*.  
**Fix 8.** `rolling(48, center=True)` uses 23 future hours. Any centred window is look-ahead.  
**Fix 9.** `rolling(24).mean()` includes the current hour. Harmless for a 24h horizon, fatal for a 1h
horizon; always write `shift(1).rolling(...)` so the feature only ever uses the past.

In [6]:
c = df["consumption_mwh"]
df["lag1"] = c.shift(1)
df["lag24"] = c.shift(24)
df["lag168"] = c.shift(168)
df["mean24"] = c.shift(1).rolling(24).mean()
df["target"] = c.shift(-24)

**Fix 10.** Never `dropna()` X and y separately and then truncate to equal length. In the original,
X lost its first 24 rows (rolling) and y did not, so `y.iloc[i]` was the consumption *at the same hour*
as `X.iloc[i]`: the model was fitted to nowcast the current hour, not forecast tomorrow. Build one frame
and drop rows once.

In [7]:
features = ["temp_c", "wind_ms", "solar_wm2", "price_eur_mwh",
            "hour", "dow", "lag1", "lag24", "lag168", "mean24"]

data = df[features + ["target"]].dropna()
print(len(data), "usable rows; first", data.index[0], "last", data.index[-1])

15833 usable rows; first 2022-01-08 11:00:00+00:00 last 2023-12-30 23:00:00+00:00


**Fix 11 / 12.** Chronological split (last 20% of time is the test set) and the scaler is fitted
on the training rows only. Shuffled hourly rows put tomorrow's neighbours in the training set.

In [8]:
split = int(len(data) * 0.8)
train, test = data.iloc[:split], data.iloc[split:]
print("train:", train.index[0].date(), "->", train.index[-1].date())
print("test :", test.index[0].date(), "->", test.index[-1].date())

scaler = StandardScaler().fit(train[features])
X_train = scaler.transform(train[features])
X_test = scaler.transform(test[features])
y_train, y_test = train["target"].values, test["target"].values

train: 2022-01-08 -> 2023-08-06
test : 2023-08-06 -> 2023-12-30


## Model

In [9]:
model = LinearRegression().fit(X_train, y_train)
pred = model.predict(X_test)

r2 = r2_score(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
print(f"model : R2 = {r2:.4f}, RMSE = {rmse:.1f} MWh")

model : R2 = 0.8611, RMSE = 1472.7 MWh


**Fix 13.** Compare against a naive baseline before reading anything into R². For a 24h-ahead target,
"same hour today" (`consumption_mwh` at t predicts the target at t+24) is the honest floor.

In [10]:
naive = c.loc[test.index].values   # consumption at t, used as the forecast for t+24
r2_naive = r2_score(y_test, naive)
rmse_naive = np.sqrt(mean_squared_error(y_test, naive))
print(f"naive : R2 = {r2_naive:.4f}, RMSE = {rmse_naive:.1f} MWh")
print(f"model beats naive RMSE by {100 * (1 - rmse / rmse_naive):.1f}%")

naive : R2 = 0.8298, RMSE = 1629.8 MWh
model beats naive RMSE by 9.6%


In [11]:
coefs = pd.Series(model.coef_, index=features).round(1).sort_values()
coefs

mean24          -1571.8
temp_c          -1413.0
dow              -185.6
wind_ms             0.8
hour              103.1
price_eur_mwh     225.2
solar_wm2         241.8
lag1              957.1
lag24            1561.3
lag168           1793.4
dtype: float64

**Fix 14.** Read the coefficient table with the horizon in mind: for a 24h-ahead target the same-hour
lags (`lag24`, `lag168`) should dominate, not `lag1`. In the broken notebook `lag1` was the top coefficient,
which was the tell that the target was actually the current hour.

## Residual check

In [12]:
res = pd.Series(y_test - pred, index=test.index)
by_hour = res.groupby(test["hour"]).agg(["mean", "std"]).round(0)
by_hour.T

hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
mean,-112.0,-179.0,-162.0,-100.0,-61.0,99.0,362.0,504.0,454.0,139.0,...,321.0,500.0,683.0,866.0,682.0,229.0,-252.0,-354.0,-432.0,-522.0
std,1542.0,1373.0,1313.0,1311.0,1370.0,1365.0,1412.0,1296.0,1337.0,1262.0,...,1462.0,1493.0,1622.0,1487.0,1410.0,1351.0,1526.0,1471.0,1587.0,1678.0


## Results (honest)

In [13]:
print(f"Held-out R2 {r2:.3f} vs naive {r2_naive:.3f}; RMSE {rmse:.0f} vs naive {rmse_naive:.0f} MWh.")
print("The model is a modest improvement over 'same hour today'. Residuals are autocorrelated and")
print("larger in the evening peak; weather forecasts (not actuals) would be the next feature to add.")

Held-out R2 0.861 vs naive 0.830; RMSE 1473 vs naive 1630 MWh.
The model is a modest improvement over 'same hour today'. Residuals are autocorrelated and
larger in the evening peak; weather forecasts (not actuals) would be the next feature to add.
